In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import altair as alt
import plotly.graph_objects as go

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving survey_results_public.csv to survey_results_public.csv


In [ ]:
# Load the dataset
df = pd.read_csv("survey_results_public.csv")

### ------------------------ 1. Learning Methods by Experience ------------------------

In [ ]:
# Clean and preprocess learning experience data
df_learn = df[['YearsCode', 'LearnCode']].dropna()
df_learn['YearsCode'] = df_learn['YearsCode'].replace({'Less than 1 year': 0, 'More than 50 years': 51}).astype(float)
df_learn['LearnList'] = df_learn['LearnCode'].str.split(';')
df_learn = df_learn.explode('LearnList')
learn_exp = df_learn.groupby('LearnList').agg(avg_exp=('YearsCode', 'mean'), count=('YearsCode', 'count')).reset_index()
learn_exp = learn_exp[learn_exp['count'] > 50]

In [ ]:
# Normalize experience for color mapping
learn_exp['norm_exp'] = (learn_exp['avg_exp'] - learn_exp['avg_exp'].min()) / (learn_exp['avg_exp'].max() - learn_exp['avg_exp'].min())

In [ ]:
# Plot interactive pictogram-style bubble plot
fig = px.scatter(
    learn_exp,
    x="LearnList",
    y="avg_exp",
    size="count",
    color="norm_exp",
    color_continuous_scale="Inferno",
    size_max=60,
    hover_name="LearnList",
    hover_data=["count", "avg_exp"]
)

# Add line plot connecting the bubbles
fig.add_trace(go.Scatter(
    x=learn_exp["LearnList"],
    y=learn_exp["avg_exp"],
    mode='lines+markers',
    line=dict(color='rgba(0, 0, 0, 0.3)', width=2),
    marker=dict(size=5, color='black'),
    showlegend=False
))

# Update layout for readability
fig.update_layout(
    title="Learning Methods vs Experience",
    xaxis_title="Learning Resource",
    yaxis_title="Average Experience (Years)",
    xaxis_tickangle=-45,
    margin=dict(t=60, l=50, r=50, b=200),
    height=600,
    template="plotly_white"
)

fig.show()

#### This chart shows how experienced developers are based on different ways they learned to code.
#### We see that people who used books or learned on the job have the highest average experience.  
#### Meanwhile, those from coding bootcamps tend to have less experience.  
#### The size and color of the bubbles help us easily compare user count and experience.

### ------------------------ 2. Top Programming Languages & Top 15 Countries ------------------------

In [ ]:
# Allow Altair to show all rows of data (no limit)
alt.data_transformers.enable('default', max_rows=None)

# Focus only on the columns for Country and Languages worked with
lang_df = df[['Country', 'LanguageHaveWorkedWith']].dropna()

# Split multiple languages (separated by ;) into separate rows
lang_df = lang_df.assign(Language=lang_df['LanguageHaveWorkedWith'].str.split(';')).explode('Language')

# Clean extra spaces from each language
lang_df['Language'] = lang_df['Language'].str.strip()

# Count how many times each language appears and get the top 15
lang_counts = lang_df['Language'].value_counts().head(15).reset_index()
lang_counts.columns = ['Language', 'Count']

# Create an interactive selection where user can click on a language
click = alt.selection_point(fields=['Language'])

# Left chart: Horizontal bar chart of top 15 languages globally
left_chart = alt.Chart(lang_counts).mark_bar().encode(
    x='Count:Q',  # number of users
    y=alt.Y('Language:N', sort='-x'),  # language names, sorted by count
    tooltip=['Language:N', 'Count:Q'],  # show tooltip on hover
    color=alt.condition(click, alt.value('#1f77b4'), alt.value('#d3d3d3'))  # highlight selected bar
).add_params(click).properties(
    width=300,
    height=400,
    title='Top Programming Languages Globally'
)

In [ ]:
# Prepare country-wise counts for each language
right_data = lang_df.groupby(['Language', 'Country']).size().reset_index(name='Count')

# Right chart: Donut chart of top 10 countries using the selected language
pie_chart = alt.Chart(right_data).transform_filter(
    click  # filter to selected language
).transform_window(
    rank='rank(Count)',  # rank countries by count
    sort=[alt.SortField('Count', order='descending')],
    groupby=['Language']
).transform_filter(
    alt.datum.rank <= 10  # show only top 10 countries
).encode(
    theta='Count:Q',  # size of slices
    color=alt.Color('Country:N', legend=alt.Legend(title="Country")),
    tooltip=['Country:N', 'Count:Q']  # tooltip on hover
).mark_arc(innerRadius=50, outerRadius=120).properties(
    width=350,
    height=400,
    title='Top 10 Country Share (Donut)'
)

# Combine the two charts side-by-side with spacing
chart = left_chart | pie_chart
chart.spacing = 50

# Show final interactive chart
chart

alt.HConcatChart(...)


### Plot Summary:
#### This interactive plot shows the top programming languages globally (bar chart on the left) and the top 10 countries using the selected language (donut chart on the right).


### Insights:
#### - JavaScript, HTML/CSS, SQL, and Python are the most commonly used languages worldwide.
#### - When a language is selected (like Python), the donut chart updates to show which countries use it the most.
#### - This helps us understand both global trends and regional preferences for programming languages.

### ------------------------ 3. Distribution of Professional Coding Experience ------------------------

In [ ]:
# Clean and filter the 'YearsCodePro' column
df = df[['YearsCodePro']].dropna()
df['YearsCodePro'] = df['YearsCodePro'].replace({
    'Less than 1 year': 0,
    'More than 50 years': 51
})
df['YearsCodePro'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')
df = df.dropna()

# Create interactive histogram + KDE using Plotly
fig = px.histogram(
    df,
    x='YearsCodePro',
    nbins=50,
    marginal="rug",  # adds individual value markers
    opacity=0.75,
    color_discrete_sequence=['#1f77b4'],
    title="Distribution of Professional Coding Experience (Interactive)"
)

fig.update_layout(
    xaxis_title="Years of Professional Coding",
    yaxis_title="Count",
    bargap=0.05,
    hovermode="x unified"
)

fig.show()


#### This plot shows how many years of coding experience developers have.
#### Most developers have 0 to 10 years of experience.
#### Very few have more than 30 years.
#### It helps us understand the general experience level in the developer community.

### ------------------------ 4. AI Task Adoption (Current vs Interested vs Not Interested) ------------------------

In [ ]:
df = pd.read_csv('survey_results_public.csv', low_memory=False, nrows=10000)
ai_cols = ['AIToolCurrently Using', 'AIToolInterested in Using', 'AIToolNot interested in Using']
ai_long = pd.DataFrame(columns=['Task', 'Status'])

for col, label in zip(ai_cols, ['Currently Using', 'Interested', 'Not Interested']):
    temp = df[[col]].dropna().copy()
    temp['Status'] = label
    temp['Task'] = temp[col].str.split(';')
    temp = temp.explode('Task')[['Task', 'Status']]
    ai_long = pd.concat([ai_long, temp], axis=0)

ai_long = ai_long.groupby(['Task', 'Status']).size().reset_index(name='Count')
top_tasks = ai_long.groupby('Task')['Count'].sum().nlargest(12).index
ai_long = ai_long[ai_long['Task'].isin(top_tasks)]

In [ ]:
status_param = alt.param(
    name='StatusSelector',
    bind=alt.binding_select(options=['All', 'Currently Using', 'Interested', 'Not Interested'], name='Filter by Status:'),
    value='All'
)

chart = alt.Chart(ai_long).mark_bar().encode(
    y=alt.Y('Task:N', sort='-x'),
    x=alt.X('Count:Q'),
    color='Status:N',
    tooltip=['Task', 'Status', 'Count']
).add_params(
    status_param
).transform_filter(
    (alt.datum.Status == status_param) | (status_param == 'All')
).properties(
    title='AI Task Adoption: Current vs Interested vs Not Interested',
    width=700
)

chart

alt.Chart(...)


#### This plot shows how developers are using AI tools across tasks.
#### Blue = Currently using, Orange = Interested, Red = Not interested.
#### Most developers use AI for writing code and searching answers.
#### Many are interested in using AI for testing and learning codebase.
#### It helps us see which tasks are common for AI use and where it's growing.

### ------------------------ 5. Average Compensation by Job Position Type ------------------------

In [ ]:
alt.data_transformers.disable_max_rows()

df = pd.read_csv("survey_results_public.csv", low_memory=False)

df_salary_clean = df[['Country', 'DevType', 'CompTotal']].dropna()
df_salary_clean['CompTotal'] = df_salary_clean['CompTotal'].astype(int)
df_salary_clean['DevType'] = df_salary_clean['DevType'].apply(lambda x: 'Other' if str(x).startswith('Other (please specify)') else x.strip())

low, high = df_salary_clean['CompTotal'].quantile([0.05, 0.95])
df_salary_filtered = df_salary_clean[(df_salary_clean['CompTotal'] >= low) & (df_salary_clean['CompTotal'] <= high)]


dropdown = alt.binding_select(
    options=[
        'United States of America',
        'Germany',
        'India',
        'United Kingdom of Great Britain and Northern Ireland',
        'Ukraine',
        'France',
        'Canada',
        'Poland',
        'Netherlands',
        'Brazil'
    ],
    name='Country: '
)

country_param = alt.param(
    value='United States of America',
    bind=dropdown,
    name='selected_country'
)

salary_bar_chart = alt.Chart(df_salary_filtered).mark_bar().encode(
    x=alt.X('DevType:N', sort='-y', title='Job Position Type'),
    y=alt.Y('CompTotal:Q', aggregate='mean', title='Average Compensation (local currency)'),
    color=alt.Color('DevType:N', legend=alt.Legend(title="Developer Type"), scale=alt.Scale(scheme='tableau20')),
    tooltip=['DevType:N',alt.Tooltip('CompTotal:Q', aggregate='mean', title='Avg Compensation')]
).add_params(
    country_param
).transform_filter(
    alt.datum.Country == country_param
).properties(
    title='Average Compensation by Job Position Type (Select Top10 Countries)',
    width=800,
    height=400
).configure_axisX(labelAngle=-45)

salary_bar_chart

alt.Chart(...)

#### In Compensation vs Job Position Type plot, we try to demostrate average compesation in different job roles.
#### We include three parts of coding to create overall visulization which are data cleaning, interactivity creation and altair bar chart.
#### First in data cleaning phase, we utilized .dropna() to eliminate the row with Na value; also, we afraid that options "Other" would create noise since survey taker could write down their position name on their own, we used lambda function that "Other (please specify)" were grouped as "Other".
#### Next, since we afraid that extreme value would influence average value, we eliminated those data that aren't inside the 5th and 95th percentile.
#### As for interactivity, we created dropdown for selecting the most data top 10 countries.